# Questão 7 - Previsão de demanda


In [125]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sqlite3
import matplotlib.pyplot as plt
import duckdb
from IPython.display import Image, display

### Caminho base do projeto

In [126]:
BASE_PATH = Path().resolve()

while BASE_PATH.name != "lh-nautical-data-project":
    BASE_PATH = BASE_PATH.parent

print(f"BASE PATH: {BASE_PATH}")

BASE PATH: /Users/richardgomes/lh-nautical-data-project


Caminhos para as pastas do projeto.

In [127]:
DATA_PATH = BASE_PATH / "data"

RAW_PATH = DATA_PATH / "raw"
STAGING_PATH = DATA_PATH / "staging"
INTERMEDIATE_PATH = DATA_PATH / "intermediate"
MARTS_PATH = DATA_PATH / "marts"

SQL_PATH = BASE_PATH / "sql"
IMAGES_PATH = BASE_PATH / "imagens"

Para descobrir o id do produto "Motor de Popa Yamaha Evo Dash 155HP"


In [128]:
df_produtos = pd.read_csv(MARTS_PATH / "dim_produto.csv")
df_produtos.head()

,id_produto,nome_produto,categoria_produto,preco_base_produto
0,1,Transponder AIS Maré Magnum,eletronicos,33122.52
1,2,Transponder Furuno Marlin,eletronicos,13998.15
2,3,Radar Furuno Pulse Leviathan,eletronicos,9024.19
3,4,Rádio AIS Hydro Tidal Zen,eletronicos,3381.88
4,5,Piloto Automático Furuno Storm,eletronicos,23669.01


In [129]:
df_produtos[
    df_produtos["nome_produto"].str.contains("Yamaha Evo Dash", case=False, na=False)
]

,id_produto,nome_produto,categoria_produto,preco_base_produto
53,54,Motor de Popa Yamaha Evo Dash 155HP,propulsao,121534.82


Série temporal por dias, considerando apenas "Motor de Popa Yamaha Evo Dash 155HP" que tem o ID 54

In [130]:
df = pd.read_csv(RAW_PATH / "vendas_2023_2024.csv")

df["sale_date"] = pd.to_datetime(df["sale_date"], format="mixed", dayfirst=True)

df = df.sort_values("sale_date")

produto_id = 54

df_produto = df[df["id_product"] == produto_id].copy()

df_diario = (
    df_produto
    .groupby("sale_date")["qtd"]
    .sum()
    .reset_index()
)

df_diario.head(10)

,sale_date,qtd
0,2023-01-10,3
1,2023-02-06,13
2,2023-02-27,15
3,2023-03-04,14
4,2023-03-15,4
5,2023-03-22,9
6,2023-04-11,4
7,2023-04-21,4
8,2023-05-09,5
9,2023-05-16,13


Corrigindo a falta de dias na série temporal criando calendário completo

In [131]:
calendario = pd.DataFrame({
    "sale_date": pd.date_range(
        start=df_diario["sale_date"].min(),
        end=df_diario["sale_date"].max(),
        freq="D"
    )
})



Um merge para juntar com as vendas

In [132]:
df_diario = calendario.merge(df_diario, on="sale_date", how="left")

Preencher dias sem venda com 0

In [133]:
df_diario["qtd"] = df_diario["qtd"].fillna(0)
df_diario["qtd"] = df_diario["qtd"].astype(int)

df_diario

,sale_date,qtd
0,2023-01-10,3
1,2023-01-11,0
2,2023-01-12,0
3,2023-01-13,0
4,2023-01-14,0
...,...,...
683,2024-11-23,0
684,2024-11-24,16
685,2024-11-25,0
686,2024-11-26,0


Separando TREINO e TESTE

In [134]:
df_treino = df_diario[df_diario["sale_date"] < "2024-01-01"].copy()
df_teste  = df_diario[df_diario["sale_date"] >= "2024-01-01"].copy()

df_treino.tail(), df_teste.head()

(     sale_date  qtd
 351 2023-12-27    0
 352 2023-12-28    0
 353 2023-12-29    0
 354 2023-12-30    0
 355 2023-12-31    0,
      sale_date  qtd
 356 2024-01-01    0
 357 2024-01-02    0
 358 2024-01-03    0
 359 2024-01-04    0
 360 2024-01-05   10)

Para criar previsão com média móvel de 7 dias

In [135]:
df_diario["media_7d"] = df_diario["qtd"].shift(1).rolling(window=7).mean()

Filtrar previsões apenas do teste

In [136]:
df_resultado = df_diario[df_diario["sale_date"] >= "2024-01-01"].copy()

df_resultado.head()

,sale_date,qtd,media_7d
356,2024-01-01,0,0.0
357,2024-01-02,0,0.0
358,2024-01-03,0,0.0
359,2024-01-04,0,0.0
360,2024-01-05,10,0.0


Calcula MAE

In [137]:
mae = (df_resultado["qtd"] - df_resultado["media_7d"]).abs().mean()

mae

np.float64(1.596815834767642)

Questão 7.2 - Validação

Utilizando seu modelo treinado, qual é a soma total da
previsão de vendas (arredondada para número inteiro)

para o Motor de Popa Yamaha Evo Dash 155HP' durante a
primeira semana de Janeiro de 2024 (01/01 a 07/01)?

Resposta: 3

Primeira semana de janeiro

In [138]:
df_semana_1 = df_resultado[
    (df_resultado["sale_date"] >= "2024-01-01") &
    (df_resultado["sale_date"] <= "2024-01-07")
]

Somando previsões

In [139]:
soma_previsao = df_semana_1["media_7d"].sum()


Arredondando para inteiro (como pedido)

In [140]:
soma_previsao_arredondada = round(soma_previsao)

soma_previsao, soma_previsao_arredondada

(np.float64(2.857142857142857), 3)

Questão 7.3 - Explique:
1. Como o baseline foi construído?
2. Como evitou data leakage?
3. Uma limitação do modelo proposto.


1. Como o baseline foi construído?

Construi o modelo usando a média móvel de 7 dias. Para cada dia do período de teste (janeiro de 2024), a previsão foi calculada como a média das vendas observadas nos 7 dias anteriores. Esse método só considera o comportamento mais recente da série temporal. Ele serve como caso de uso simples, apenas uma referência de previsão da demanda.

2. Como evitou data leakage?

Evitei o data leakage garantindo que, para cada previsão, fossem utilizados apenas dados históricos anteriores à data prevista, usando da função shift(1).

3. Uma limitação do modelo proposto

Esse tipo de modelo não captura padrões mais complexos, como tendências de longo prazo, sazonalidade ou fatores externos (como promoções ou mudanças de mercado). Por ser baseado apenas na média recente.